In [ ]:
from thbsplines.hierarchical_space import HierarchicalSpace
import numpy as np
import scipy.sparse as sp
import dolfinx
from mpi4py import MPI
import basix.ufl
import pyvista


from dolfinx import default_real_type, default_scalar_type
rtype = default_real_type
dtype = default_scalar_type
import ufl


from thbsplines.refinement import refine
from thbsplines.fenicsx.mesh import build_mesh, FastMidpointMapper
from thbsplines.fenicsx.functionspace import build_dofmap, fill_function_space, create_spline_space
from thbsplines.fenicsx.solvers import solve_problem, enforce_dirichlet_boundary
from thbsplines.fenicsx.adaptivity import dorfler_marking
from thbsplines.fenicsx.kernels import make_linear_kernel, make_bilinear_kernel
from thbsplines.fenicsx.postprocessing import map_spline_to_legendre
from thbsplines.fenicsx.forms import mark_cells, make_bilinear_form, make_linear_form

In [ ]:
n_refinements = 1
p0 = 3
knots1 = np.array([0,0,0, 0.5,0.5,1, 1, 1], dtype=np.float64)
#knots1 = np.array([-1, -1, -1, 0,0,1,1,1], dtype=np.float64)
knots1 = refine(knots1, p=p0, n_times=n_refinements)
log_initial_mesh_size = np.log2(np.max(np.diff(knots1)))
knots2 = np.array([0,0,0,0.5,1,1,1], dtype=np.float64)
#knots2 = np.array([-1, -1, -1, 0,0,1,1,1], dtype=np.float64)
knots2 = refine(knots2, p0, n_times=n_refinements-1)
err_cells = {}
hs = HierarchicalSpace(knots=[knots1, knots2], degrees=[p0])


In [ ]:
for level, cells in err_cells.items():
    #print(cells)
    hs.refine(cells, level, refine_neighbours=False, refine_T_neighbours=True, m=3)
hs.hmesh.plot_cells()

In [ ]:
def map_uv_to_xy_small(uv_points, nodes_per_cell=4):
    original_shape = uv_points.shape
    uv_flat = uv_points.reshape(-1, 2)
    
    u = uv_flat[:, 0]
    v = uv_flat[:, 1]
    xy_flat = np.zeros_like(uv_flat)

    # ---------------------------------------------------------
    # MACRO PATCH 1: Left Half (u < 0.5)
    # ---------------------------------------------------------
    left_mask = u < 0.5
    u_L = u[left_mask]
    v_L = v[left_mask]
    # P(U,V) = (1-U)(1-V)P00 + U(1-V)P10 + (1-U)VP01 + UVP11
    # for this specific problem, 
    # P00=(0,-1), 
    # P10 =(0,0)
    # P01 = (-1, -1), 
    # P11 = (-1, 1)
    
    xy_flat[left_mask, 0] = -v_L
    # substituting U=2u, V=v
    xy_flat[left_mask, 1] = 2 * u_L * v_L + 2 * u_L - 1

    # ---------------------------------------------------------
    # MACRO PATCH 2: Right Half (u >= 0.5)
    # ---------------------------------------------------------
    right_mask = u >= 0.5
    u_R = u[right_mask]
    v_R = v[right_mask]
    
    xy_flat[right_mask, 0] = 2 * u_R * v_R + 2 * u_R - 2 * v_R - 1
    xy_flat[right_mask, 1] = v_R
    
    return xy_flat.reshape(original_shape)


disconnected_mesh, thb_operators, N_max, physical_cells_midpoints = build_mesh(hs=hs, mapping=map_uv_to_xy_small)

In [ ]:
# topology, cell_types, geometry = dolfinx.plot.vtk_mesh(disconnected_mesh)
# grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# plotter = pyvista.Plotter()
# plotter.add_mesh(grid.shrink(.95), show_edges=False, color="#03bb85")
# plotter.view_xy()
# plotter.show(jupyter_backend="static")
# print(f"Number of points in PyVista grid: {grid.n_points}")

In [ ]:
def outer_boundary(x):
    on_left = np.isclose(x[0], -1.)
    on_bottom = np.isclose(x[1], -1.)
    on_right = np.isclose(x[0], 1.) & (x[1]>=-1e-10)
    on_top = np.isclose(x[1], 1.) & (x[0]<=1.)
    return on_left|on_bottom|on_right|on_top

legendre_elt = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0,
    lagrange_variant=basix.LagrangeVariant.legendre
)
V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")
facet_dim = disconnected_mesh.topology.dim-1
boundary_facets = dolfinx.mesh.locate_entities_boundary(disconnected_mesh, facet_dim, outer_boundary)
custom_metadata = {"quadrature_degree": 6}
#ds = ufl.Measure("ds", domain=disconnected_mesh, subdomain_data=([1, boundary_facets]), metadata=custom_metadata)
dx_custom = ufl.Measure("dx", domain=disconnected_mesh, metadata=custom_metadata)


u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
my_x = ufl.SpatialCoordinate(disconnected_mesh)
# f = dolfinx.fem.Function(V)
#f.interpolate(lambda x: x[0]*x[1]+0.9*x[0]**2-.7)
f =  1./(1.*ufl.exp((my_x[0]+.0625)**2 + (my_x[1]-.0625)**2))
a0 = ufl.inner(u,v)*dx_custom
f0 = ufl.inner(f,v)*dx_custom
f_square_integral = dolfinx.fem.assemble_scalar(dolfinx.fem.form(ufl.inner(f, f)*dx_custom))
f_sq_integral = np.sqrt(disconnected_mesh.comm.allreduce(f_square_integral, op=MPI.SUM))

msh = disconnected_mesh

In [ ]:
dofmap, padded_cells_to_dofs = build_dofmap(hierarchical_space=hs, mesh=disconnected_mesh, 
                                            N_max=N_max, morton=False)
custom_mapping = FastMidpointMapper(hs, physical_cells_midpoints)
C_func, C_space = fill_function_space(hierachical_space=hs, mesh=disconnected_mesh,
                                      N_max=N_max, thb_operators=thb_operators, mapping_function=custom_mapping)
V_spline = create_spline_space(cells_to_dofs=padded_cells_to_dofs, mesh=disconnected_mesh,
                               N_max=N_max, mult_factor=1)

local_dofs = (hs.degrees[0]+1)**2
tabulate_A = make_bilinear_kernel(disconnected_mesh, a0, padded_dofs=N_max, local_dofs=local_dofs)
tabulate_L = make_linear_kernel(disconnected_mesh, f0, padded_dofs=N_max, local_dofs=local_dofs)

cell_domain = mark_cells(mesh=disconnected_mesh)
a_cond = make_bilinear_form(mesh=disconnected_mesh,
                            ufl_form=a0,
                            trial_space=V_spline, test_space=V_spline,
                            coefficients=C_func, integrals=[(cell_domain, tabulate_A)])
l_cond = make_linear_form(mesh=disconnected_mesh, 
                          ufl_form=f0,
                          test_space=V_spline,
                          coefficients=C_func,
                          integrals=[(cell_domain, tabulate_L)])

In [ ]:
forbidden_indices = enforce_dirichlet_boundary(hs, dofmap, bottom=True)
x_vec, A = solve_problem(hs=hs, a=a_cond, rhs=l_cond, dirichlet_indices=forbidden_indices,
                         dummy_index=np.max(padded_cells_to_dofs), V_spline=V_spline,
                         iterative=False, return_A=True)

u_dg = map_spline_to_legendre(hs, V, C_func, N_max, disconnected_mesh, padded_cells_to_dofs, x_vec)
u_dg.x.scatter_forward()


# Compute exact L2 error using FEniCSx standard UFL
error_form = dolfinx.fem.form(ufl.inner(f - u_dg, f - u_dg) * dx_custom)
error_sq = dolfinx.fem.assemble_scalar(error_form)
exact_l2_error = np.sqrt(disconnected_mesh.comm.allreduce(error_sq, op=MPI.SUM))

print(f"Exact L2 Error (via DG projection): {exact_l2_error:.4e}")
print(f"Relative error = {exact_l2_error/f_sq_integral:.4e}")

In [ ]:
# Create a DG0 space (one value per cell)
V_error = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0))
v = ufl.TestFunction(V_error)
hQ = ufl.CellDiameter(disconnected_mesh)
volume_form = dolfinx.fem.form(1.0*v*dx_custom)
cell_volumes = dolfinx.fem.assemble_vector(volume_form).array
#print(f"cell_volumes = {cell_volumes[:10]}")
# Define the local L2 error form: integral of (f - u_dg)^2 per cell
# Note: We multiply by the test function 'v' to pick out each cell's contribution
local_error_form = dolfinx.fem.form(ufl.inner(f - u_dg, f - u_dg) *v * dx_custom)
err_cells = dorfler_marking(hs, 0.5, local_error_form=local_error_form)